# Spatial Feature Pipeline Summary

This notebook summarizes the spatial feature identification pipeline from tracked files only.

In [ ]:
from pathlib import Path
import collections
import re
import subprocess
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def git_files():
    result = subprocess.run(["git", "ls-files"], cwd=ROOT, text=True, capture_output=True)
    if result.returncode != 0:
        display(Markdown(f"**Tracked file inventory not available:** `{result.stderr.strip()}`"))
        return []
    return [line.strip().replace("\\", "/") for line in result.stdout.splitlines() if line.strip()]


def read_tracked(path, limit=6000):
    path = path.replace("\\", "/")
    files = set(git_files())
    if path not in files:
        return ""
    full = ROOT / path
    try:
        return full.read_text(encoding="utf-8", errors="replace")[:limit]
    except Exception as exc:
        return f"Could not read {path}: {exc}"


def tracked_exists(path):
    return path.replace("\\", "/") in set(git_files())


def tracked_matching(*patterns):
    files = git_files()
    out = []
    for path in files:
        low = path.lower()
        if all(pattern.lower() in low for pattern in patterns):
            out.append(path)
    return out


def md(lines):
    if isinstance(lines, str):
        display(Markdown(lines))
    else:
        display(Markdown("\n".join(lines)))


def bullet(paths, limit=40):
    paths = list(paths)
    if not paths:
        return ["- Not available in tracked repository."]
    shown = paths[:limit]
    lines = [f"- `{p}`" for p in shown]
    if len(paths) > limit:
        lines.append(f"- ... {len(paths) - limit} more tracked files omitted from this display")
    return lines

In [ ]:
files = git_files()
paths = [p for p in files if p.startswith("spatial_feature_identification_pipeline/")]
lines = ["## Tracked Spatial Feature Pipeline Files", ""]
lines.extend(bullet(paths, limit=80))
md(lines)

In [ ]:
text_sources = [p for p in git_files() if p.startswith("spatial_feature_identification_pipeline/") and p.lower().endswith((".md", ".txt", ".yaml", ".yml", ".json"))]
combined = "\n".join(read_tracked(p, limit=12000) for p in text_sources)
expected = {
    "102 retained Visium samples": ["102", "retained"],
    "661 numeric spatial architecture features": ["661", "feature"],
    "12 source datasets": ["12", "dataset"],
}
lines = ["## Tracked Documentation Numbers", ""]
for label, terms in expected.items():
    found = all(term.lower() in combined.lower() for term in terms)
    lines.append(f"- {label}: {'found in tracked documentation' if found else 'not available in tracked repository text'}")
lines.extend(["", "## Role", "", "The spatial feature pipeline converts staged spatial omics/Visium inputs into numeric sample-level architecture features and manifests used by downstream teacher-building and spatial prediction modules."])
md(lines)